# CLIFFGUARD - round 4: the whole ladder under the corrected scorer

Use a T4 GPU, then Run all. Expect about **1 hour 40 minutes**.

Round 3 established that the original label scorer was comparing
incommensurable logits, and re-graded enough of the ladder to test the headline
4.5-bit comparison: full precision and the 4.5-bit rung on the two behavioural
runs, full precision on the three labelled ones. That is 2 of 8 rungs and 1 of
8.

Every quantity defined over the *whole* ladder is therefore still an
original-scorer quantity: the drift coefficient and its bootstrap, the 14-cell
transition table, the simultaneous one-sided bound, the refusal-law figure, and
all 21 labelled model-by-rung cells.

This notebook grades the remaining rungs. It generates nothing. Every
completion it reads is already in the repository, so there is no corpus to
rebuild, no model under test to download, and nothing to lose if the session
dies -- each grading checkpoints to Drive as it finishes.

**15,200 judge pairs.** 8,000 three-way (2 models x 8 rungs x 500 prompts) and
7,200 five-way (3 models x 8 rungs x 300 prompts), at the ~178 pairs/minute this
judge sustains at batch 8 on a T4.


## Environment

The setup is deliberately the same as round 2. It mounts Drive before any model work, so completed schemes survive a Colab disconnect.

In [ ]:
import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/parnish007/CLIFFGUARD.git'
# Set to the commit this notebook was validated against; '' follows main.
REPO_COMMIT = '95674fef58bddd4081366e6ec8f8feb27f37d51f'
REPO_DIR = pathlib.Path('/content/CLIFFGUARD') if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/cliffguard')

if IN_COLAB:
    # Fatal, not a warning. Without Drive every cache and run directory lives
    # on disk Colab wipes at disconnect, so an unattended session that loses
    # its connection at hour two loses the whole session. Continuing without it
    # is not a degraded run, it is a run that cannot survive the thing most
    # likely to happen to it.
    from google.colab import drive as _drive
    _drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    # Pinned to the commit this notebook was written against. A clone of
    # whatever main happens to be would silently run different analysis code,
    # and --depth 1 cannot reach a specific commit, hence the full clone above.
    if REPO_COMMIT:
        subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], check=True)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                          text=True, check=True).stdout.strip()
    print(f'repo commit  : {head}')
    if REPO_COMMIT and not head.startswith(REPO_COMMIT):
        raise SystemExit(f'checked out {head}, expected {REPO_COMMIT}')

    # check=True: a missing bitsandbytes surfaces as a CUDA error inside the
    # judge two hours from now rather than here.
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'bitsandbytes', 'datasets', 'accelerate'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch, numpy as np, transformers
HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else 'NONE'
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0
print(f'repo         : {pathlib.Path.cwd()}')
print(f'python       : {platform.python_version()}')
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {np.__version__}')
print(f'GPU          : {GPU_NAME}  ({VRAM_GB} GB)')
if hasattr(os, 'statvfs'):
    st = os.statvfs('.')
    print(f'free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB')
if not HAS_GPU:
    raise SystemExit('No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.')
if tuple(int(p) for p in transformers.__version__.split('.')[:2]) < (4, 45):
    raise SystemExit(f'transformers {transformers.__version__} too old (need >= 4.45).\nRun: !pip -q install -U transformers, then restart the runtime.')


## Restore

Drive state first, so the preflight below validates what will actually be used.

In [ ]:
# Restore Drive state BEFORE validating it. On a fresh Colab clone `data/` does
# not exist -- it is gitignored -- so the corpora can only come from Drive or a
# rebuild, and the run directories from a previous session likewise. Validating
# first would fail on files that were about to arrive, or pass a check on files
# that were never going to.
import shutil

def _restore(src: pathlib.Path, dst: pathlib.Path, what: str) -> bool:
    if not src.exists():
        print(f'[drive] no {what} to restore')
        return False
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[drive] restored {what} -> {dst}')
    return True

_restore(DRIVE_ROOT / 'fold_a', pathlib.Path('data/folds/fold_a'), 'Fold A corpus')
_restore(DRIVE_ROOT / 'eval_suites', pathlib.Path('data/eval_suites'), 'eval suites')
_restore(DRIVE_ROOT / 'artifacts' / 'runs', pathlib.Path('artifacts/runs'),
         'previous run directories')

# The five published runs now ship WITH the repository, so the clone already
# has them and step 3 needs no upload. Verified rather than assumed: their
# absence would make the re-grade skip silently, and that is the step whose
# whole purpose is to check the published numbers.
_published = ['colab-behavioural-qwen3b', 'colab-behavioural-phi35',
              'lab-qwen3b-xstest', 'lab-phi35-xstest', 'lab-smol17-xstest']
_found = [p for p in _published
          if list(pathlib.Path('artifacts/runs').glob(f'*{p}'))]
print(f'[repo] published runs available: {len(_found)}/{len(_published)}')
if len(_found) < len(_published):
    print(f'       MISSING {sorted(set(_published) - set(_found))} -- the '
          'step-3 re-grade will skip those and report them as not run')

# A Drive zip still overrides, for anyone re-running with different data.
_prior = DRIVE_ROOT / 'prior_runs.zip'
if _prior.exists():
    import zipfile
    with zipfile.ZipFile(_prior) as zf:
        zf.extractall('.')
    print(f'[drive] {_prior.name} unpacked over the repository copies '
          f'({len(zf.namelist())} files)')

# Rebuild only what is still missing. Fold A comes from an UNPINNED HuggingFace
# revision, so a rebuild can silently produce a different corpus and break the
# pairing with the 48-token runs; preflight hashes it immediately afterwards,
# which is what turns that risk into a caught error rather than a wasted
# session. XSTest is a single file at a stable URL and is safer to fetch.
if not pathlib.Path('data/folds/fold_a/anthropic_hh_refused.jsonl').exists():
    print('[corpus] Fold A missing; rebuilding (preflight will verify the hash)')
    subprocess.run([sys.executable, 'scripts/download_fold_a.py', '--download'],
                   check=True)
if not pathlib.Path('data/eval_suites/xstest.jsonl').exists():
    print('[corpus] XSTest missing; fetching')
    subprocess.run([sys.executable, 'scripts/download_eval_suites.py',
                    '--download', '--suites', 'xstest'], check=False)

# Cache the corpora back, so the next session restores instead of re-fetching.
if DRIVE_ROOT.exists():
    for src, name in ((pathlib.Path('data/folds/fold_a'), 'fold_a'),
                      (pathlib.Path('data/eval_suites'), 'eval_suites')):
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / name, dirs_exist_ok=True)
    print('[drive] corpora cached')

## Preflight

This gate runs before any expensive model load. Do not start the measurements if it reports a failure.

In [ ]:
# --require-xstest because this notebook's second half needs the labelled
# corpus. Without the flag a missing file is only a `skip`, preflight passes,
# and the failure surfaces after the two expensive HH-RLHF steps have already
# spent their GPU time.
preflight = subprocess.run([sys.executable, 'scripts/preflight_round2.py',
                            '--require-xstest'])
if preflight.returncode != 0:
    raise SystemExit(
        'PREFLIGHT FAILED. Do not start round 3; repair the reported gate '
        'failure first. Every check above runs on CPU in seconds, so fixing '
        'it costs nothing compared with discovering it at hour three.')

In [ ]:
MODELS_LONG = [('qwen3b', 'Qwen/Qwen2.5-3B-Instruct'), ('phi35', 'microsoft/Phi-3.5-mini-instruct')]
MODELS_XSTEST = MODELS_LONG + [('smollm17b', 'HuggingFaceTB/SmolLM2-1.7B-Instruct')]
JUDGE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
N_LONG, N_XSTEST, LONG_TOKENS, SEED = 250, 150, 256, 0
JUDGE_COMPLETION_CHARS, TAXONOMY_MAX_LENGTH = 2000, 2560
# Five verified single-token options rather than label-word prefixes.
# Under Qwen2.5 the old mode compared ' REF', ' COM', ' DEF', ' UNC'
# -- three-character prefixes shared with common words -- against
# ' DISCLAIM', an entire word. The five logits were therefore not
# commensurable, and DISCLAIM, the rarest observed class, was the one
# scored differently from the other four.
SCORING = 'letter'

# Caches go straight to Drive when it is mounted. The old mirror-between-arms
# design left completed schemes on Colab's disposable disk until a whole model
# had finished, so a disconnect could lose hours of valid cache entries.
CACHE_ROOT = (DRIVE_ROOT / 'artifacts') if DRIVE_ROOT.exists() else pathlib.Path('artifacts')
BEHAV_CACHE = str(CACHE_ROOT / 'behavioural_cache')
RESULTS = {}
# Verified by preflight; repeated here so the resume path can reject a
# restored directory whose prompts are not the ones we are pairing against.
FOLD_A_SHA = '7da25bf88ee0409ce4900a12052e15849a2898ed01cfdcdfe6409bbfc11bd9b5'
XSTEST_SHA = '33874ac77bd574a74283cd024466f442e69da870fa1195fcde8a9107433f9ce4'
print(f'long HH-RLHF : {N_LONG} per class, {LONG_TOKENS} tokens, FP16 + RTN 4-bit')
print(f'XSTest       : {N_XSTEST} per class, {LONG_TOKENS} tokens, FP16 only')
print(f'caches       : {CACHE_ROOT}' + ('' if DRIVE_ROOT.exists() else '   (LOCAL -- a disconnect loses them)'))

def run_step(label, script, args, timeout=10800):
    '''Stream one script invocation; keep its tail and exit status.

    The timeout is a watchdog rather than proc.wait(timeout=...). Reading a
    child's stdout to EOF blocks for as long as it lives, so wait was reached
    only after exit and could never stop a hung step. A watchdog kill is kept
    separate because Linux reports it as -9, the same code as an OOM kill.
    '''
    import threading
    cmd = [sys.executable, f'scripts/{script}'] + args
    print(f'\n$ {" ".join(cmd)}', flush=True)
    started, lines = time.time(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timed_out = []
    def _kill():
        timed_out.append(True)
        print(f'\n[{label}] no exit after {timeout / 3600:.1f} h; killing', flush=True)
        proc.kill()
    watchdog = threading.Timer(timeout, _kill)
    watchdog.daemon = True
    watchdog.start()
    try:
        for line in proc.stdout:
            if 'Loading weights' in line or 'it/s]' in line or 's/prompt' in line:
                continue
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait()
    except Exception as exc:
        proc.kill()
        proc.wait()
        lines.append(f'ABORTED: {type(exc).__name__}: {exc}')
    finally:
        watchdog.cancel()
    ok = proc.returncode == 0
    RESULTS[label] = {'returncode': proc.returncode, 'timed_out': bool(timed_out), 'minutes': (time.time() - started) / 60, 'tail': lines[-40:]}
    print(f'\n=== {label}: {"OK" if ok else f"FAILED rc={proc.returncode}"} in {RESULTS[label]["minutes"]:.1f} min ===', flush=True)
    return ok

def run_step_resumable(label, script, args, attempts=3, timeout=10800):
    '''Retry only a real OOM. Each completed scheme is cached immediately, so a
    fresh process resumes farther through the ladder instead of starting over.
    A bad flag, missing checkpoint, or timeout cannot be improved by retrying.'''
    tag = label
    for attempt in range(1, attempts + 1):
        tag = label if attempt == 1 else f'{label}-retry{attempt}'
        if attempt > 1:
            print(f'\n[retry {attempt}/{attempts}] {label}: resuming from cache', flush=True)
        if run_step(tag, script, args, timeout=timeout):
            RESULTS[label] = RESULTS[tag]
            return True
        result = RESULTS[tag]
        if result['returncode'] != -9 or result['timed_out']:
            reason = 'timed out' if result['timed_out'] else f'rc={result["returncode"]}'
            print(f'[{label}] {reason} is not a resumable OOM; not retrying', flush=True)
            RESULTS[label] = result
            return False
    print(f'[{label}] still failing after {attempts} attempts', flush=True)
    RESULTS[label] = RESULTS[tag]
    return False

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # GPU allocation is released between schemes; host memory ratchets upward
    # across model loads and is what eventually triggers Colab's OOM killer.
    host = ''
    try:
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                host = f', host available {float(line.split()[1]) / 1e6:.1f} GB'
    except OSError:
        pass
    print(f'[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated{host}')

def checkpoint_to_drive():
    '''Mirror completed run directories to Drive; caches already live there.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ('runs', 'behavioural_cache', 'sector_cache'):
        src = pathlib.Path('artifacts') / name
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / 'artifacts' / name, dirs_exist_ok=True)
    print(f'[drive] mirrored artifacts/ to {DRIVE_ROOT}')

def restore_from_drive():
    '''Bring back prior run directories before anything runs.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    src = DRIVE_ROOT / 'artifacts' / 'runs'
    if src.exists():
        shutil.copytree(src, pathlib.Path('artifacts') / 'runs', dirs_exist_ok=True)
        print('[drive] restored artifacts/runs')


def latest_run(pattern):
    hits = sorted(pathlib.Path('artifacts/runs').glob(pattern))
    return hits[-1] if hits else None

def completed_run(label, schemes, model, n_prompts, corpus_sha=None,
                  tokens=None):
    '''A restored run directory that is genuinely finished, not merely present.

    Existence is not completion. A Drive copy interrupted mid-write, a stale
    directory from a run with different arguments, or a truncated JSON all look
    identical to `path.exists()`, and treating any of them as done would skip
    the step and hand the analysis silently wrong data. That is worse than
    re-running: a missing result is visible, a wrong one is not.

    So everything the step depends on is checked against the manifest --
    model, scheme list, prompt count, token budget, seed, and the ordered
    corpus hash that makes the comparison paired at all -- and every
    completions file is parsed and counted rather than stat-ed.
    '''
    tokens = LONG_TOKENS if tokens is None else tokens
    complete = []
    for run in sorted(pathlib.Path('artifacts/runs').glob(f'*_{label}')):
        try:
            manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
        except (OSError, ValueError):
            print(f'[resume] {run.name}: unreadable manifest; ignoring')
            continue
        expected = {'model_id': model, 'n_prompts': n_prompts,
                    'max_new_tokens': tokens, 'seed': SEED}
        wrong = {k: (manifest.get(k), v) for k, v in expected.items()
                 if manifest.get(k) != v}
        if wrong:
            print(f'[resume] {run.name}: arguments differ {wrong}; ignoring')
            continue
        if list(manifest.get('schemes', [])) != schemes:
            print(f'[resume] {run.name}: schemes {manifest.get("schemes")} '
                  f'!= {schemes}; ignoring')
            continue
        digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
        if corpus_sha and digest != corpus_sha:
            print(f'[resume] {run.name}: corpus hash differs; ignoring')
            continue
        ok = True
        for scheme in schemes:
            path = run / 'results' / f'completions_{scheme}.json'
            try:
                blob = json.loads(path.read_text(encoding='utf-8'))
                texts = blob['completions'] if isinstance(blob, dict) else blob
            except (OSError, ValueError, KeyError):
                print(f'[resume] {run.name}: {path.name} missing or unparseable')
                ok = False
                break
            if len(texts) != n_prompts:
                print(f'[resume] {run.name}: {path.name} has {len(texts)} rows, '
                      f'expected {n_prompts}')
                ok = False
                break
        if ok:
            complete.append(run)
    if len(complete) > 1:
        raise SystemExit(f'multiple completed runs share {label}: '
                         f'{[p.name for p in complete]}. Refusing to choose one.')
    return complete[0] if complete else None


def grade_complete(run, filename, schemes, n_prompts):
    '''A grading output that covers every scheme and every prompt.

    Same argument as above: a half-written grade file exists just as much as a
    finished one, and skipping the grader on the strength of that would leave
    the analysis reading rows that were never produced.
    '''
    path = run / 'results' / filename
    try:
        blob = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return False
    # The scorer is part of what makes a grading the right grading. A file
    # written under first-token scoring satisfies every structural check here
    # while being the instrument this round exists to replace, so skipping on
    # its presence would silently keep the old measurement.
    if blob.get('scoring') not in (None, SCORING):
        print(f"[resume] {path.name}: scored under "
              f"{blob.get('scoring')!r}, not {SCORING!r}; will re-grade")
        return False
    results = blob.get('results') or blob.get('verdicts') or {}
    missing = [s for s in schemes if s not in results]
    if missing:
        print(f'[resume] {path.name}: no rows for {missing}; will re-grade')
        return False
    return True


def print_pairing(run):
    """Show the ordered corpus hash, so pairing is visible not assumed."""
    manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
    digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
    print(f'[run] {run}')
    print(f'[pairing] corpora.prompts.sha256_ordered = {digest}')


def record_skip(label, reason):
    RESULTS[label] = {'returncode': 0, 'skipped': True, 'minutes': 0.0,
                      'tail': [reason]}
    print(f'[{label}] already complete after Drive restore; skipping: {reason}')


## The re-grade

Five runs, every rung, letter scoring. The `--schemes` list is written out
rather than left to default, because the default grades whatever the manifest
happens to name and these run directories carry completions the published
analysis never scored.

Two things are deliberately held identical to round 3's partial re-grade, so
the rungs graded there and the rungs graded here are one measurement and not
two: batch size 8, and the same 2000-character completion window. The window is
inert on this text -- the longest 48-token completion in any of these runs is
465 characters -- but holding it fixed costs nothing and removes a question.

Full precision is re-graded here as well, even though round 3 already did it.
It is a few minutes, and it makes each run's ladder internally complete under
one fingerprint rather than assembled from two sessions.

In [ ]:
# Every rung, not just the two that carry the headline. The point of this
# notebook is the quantities that are defined over the whole ladder, and those
# cannot be assembled from a partial re-grade.
ALL_RUNGS = ['FP16', 'RTN_8B', 'RTN_7B', 'RTN_6B', 'RTN_5B', 'RTN_4B',
             'RTN_3B', 'RTN_2B']

REGRADE = [
    ('qwen3b',        '*colab-behavioural-qwen3b', 500, 'classify_completions_judge.py'),
    ('phi35',         '*colab-behavioural-phi35',  500, 'classify_completions_judge.py'),
    ('qwen3b-xstest', '*lab-qwen3b-xstest',        300, 'classify_completion_taxonomy.py'),
    ('phi35-xstest',  '*lab-phi35-xstest',         300, 'classify_completion_taxonomy.py'),
    ('smol17-xstest', '*lab-smol17-xstest',        300, 'classify_completion_taxonomy.py'),
]

missing = []
for tag, pattern, n_prompts, script in REGRADE:
    run_dir = latest_run(pattern)
    if run_dir is None:
        missing.append(tag)
        continue

    # Grade only the rungs whose completions are actually present. A run that
    # stops short of the full ladder is not a failure, and naming a scheme with
    # no completions file would abort the whole grading rather than skip it.
    present = [s for s in ALL_RUNGS
               if (run_dir / 'results' / f'completions_{s}.json').exists()]
    if not present:
        missing.append(f'{tag} (no completions)')
        continue

    label = f'r4-{tag}'
    free_vram()
    args = [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
            '--completion-chars', str(JUDGE_COMPLETION_CHARS),
            # Held identical to round 3's partial re-grade so the rungs graded
            # there and the rungs graded here form one measurement. Batch size
            # is not cosmetic: fp16 kernels reduce in a batch-dependent order
            # and a near-tied pair of label logits can cross, which is exactly
            # what this project spent round 3 measuring.
            '--batch-size', '8',
            '--scoring', 'letter',
            '--schemes', *present]
    if script == 'classify_completion_taxonomy.py':
        args += ['--max-length', str(TAXONOMY_MAX_LENGTH)]

    print(f'\n{label}: grading {run_dir.name}')
    print(f'  {len(present)} rungs x {n_prompts} prompts = '
          f'{len(present) * n_prompts} pairs under letter scoring')
    run_step_resumable(label, script, args, timeout=60 * 60)
    checkpoint_to_drive()

if missing:
    print(f'\nNOT RE-GRADED: {missing}')
    print('These runs ship with the repository, so a missing one means the '
          'clone or the Drive restore is incomplete rather than that the data '
          'does not exist. Every ladder-wide quantity stays an original-scorer '
          'quantity until they are graded.')
else:
    print('\nevery rung of all five published runs graded under the corrected scorer')


## What is now available

A count rather than a claim. The cell below reports, per run, how many rungs
carry a corrected-scorer cache, so the answer to *can the ladder-wide
quantities be recomputed* is a number on the screen and not an inference from
the absence of an error.

In [ ]:
from cliffguard.eval.scorer_caches import resolve, resolve_taxonomy

print(f"{'run':38s} {'scorer':14s} {'rungs':>5s}")
print('-' * 62)
complete = True
for tag, pattern, n_prompts, script in REGRADE:
    run_dir = latest_run(pattern)
    if run_dir is None:
        continue
    if script == 'classify_completions_judge.py':
        found = resolve(run_dir, completion_chars=600)
        prefix = 'judge'
    else:
        found = resolve_taxonomy(run_dir)
        prefix = 'taxonomy'
    digest = found.get('letter')
    rungs = sorted(p.stem.split('_', 2)[-1]
                   for p in run_dir.glob(f'results/{prefix}_{digest}_*.json')) if digest else []
    print(f'{run_dir.name[-38:]:38s} {"letter":14s} {len(rungs):5d}')
    if len(rungs) < 8:
        complete = False

print()
if complete:
    print('Every published run now has all eight rungs under the corrected '
          'scorer. The drift coefficient, the transition table, the '
          'simultaneous bound and the 21 labelled cells can all be recomputed.')
else:
    print('At least one run is short of the full ladder. Ladder-wide quantities '
          'must stay labelled as original-scorer results until it is not.')


## Export

The archive carries the re-graded run directories. Activations are excluded:
they are 63 to 95 MB each, nothing here reads them, and the probe refit that
does read them runs locally on a CPU.

As in round 3, the filename is prefixed `INCOMPLETE_` if any grading failed or
never ran, because a step that produced nothing is not a step that found
nothing.

In [ ]:
import zipfile

stamp = time.strftime('%Y%m%d-%H%M%S')
failed = [k for k, v in RESULTS.items()
          if v.get('returncode') not in (0, None) or v.get('blocked')]
never_ran = [f'r4-{tag}' for tag, *_ in REGRADE
             if f'r4-{tag}' not in RESULTS]
prefix = 'INCOMPLETE_' if (failed or never_ran) else ''
archive = pathlib.Path(f'/content/{prefix}cliffguard_round4_{stamp}.zip')

status = {'steps': RESULTS, 'failed': failed, 'never_ran': never_ran,
          'scoring': 'letter', 'rungs': ALL_RUNGS,
          'completion_chars': JUDGE_COMPLETION_CHARS,
          'taxonomy_max_length': TAXONOMY_MAX_LENGTH, 'batch_size': 8}

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for tag, pattern, n_prompts, script in REGRADE:
        run_dir = latest_run(pattern)
        if run_dir is None:
            continue
        for path in sorted(run_dir.rglob('*')):
            if not path.is_file() or 'activations' in path.parts:
                continue
            zf.write(path, str(path.relative_to(REPO_DIR)))
    zf.writestr('artifacts/runs/ROUND4_STATUS.json', json.dumps(status, indent=2))

print(f'wrote {archive}  ({archive.stat().st_size / 1e6:.1f} MB)')
if failed or never_ran:
    print(f'INCOMPLETE: failed={failed} never_ran={never_ran}')
else:
    print('all gradings completed')

# To Drive as well: an unattended browser download is not durable.
try:
    shutil.copy2(archive, DRIVE_ROOT / archive.name)
    print(f'copied to Drive: {archive.name}')
except Exception as error:
    print(f'could not copy to Drive ({error}); download it from /content')

try:
    from google.colab import files
    files.download(str(archive))
except Exception:
    pass
